# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer Exploration with `mlcroissant`
This notebook demonstrates loading, exploring, and processing the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset is defined via a Croissant schema URL:

```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'
# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print('Dataset Name:', metadata.name)
print('Description:', metadata.description)
print('Version:', metadata.version)
print('Published:', metadata.datePublished)
print('Citation:', metadata.citeAs)
print('\nKeywords:', metadata.keywords)
print('\nPersonal Sensitive Information:', metadata.personalSensitiveInformation)

# Optionally, pretty-print the full metadata object for detail
# pprint.pprint(vars(metadata))

## 2. Data Overview
Review available record sets, their fields, columns, and unique `@id`s. All references use the entity `@id`.

In [ ]:
# Discover record sets

record_sets = dataset.record_sets()

if not record_sets:
    print("No record sets found in the Croissant schema. Please check the dataset definition.")
else:
    print("Record Sets found:")
    for rs in record_sets:
        print(f"  Name: {rs.name}\n  @id: {rs.id}\n  Description: {rs.description}")

    # For each record set, print available fields and columns by @id
    for rs in record_sets:
        print(f"\nFields for Record Set '{rs.name}' (@id: {rs.id}):")
        if hasattr(rs, 'fields') and rs.fields:
            for fld in rs.fields:
                print(f"  Field: {getattr(fld, 'name', '<no name>')} | @id: {fld.id} | Description: {getattr(fld, 'description', '')}")
                # List columns (if any)
                if hasattr(fld, 'columns') and fld.columns:
                    for col in fld.columns:
                        print(f"   - Column: {col.name} | @id: {col.id} | Type: {getattr(col, 'data_type', '')}")
        else:
            print("  No fields defined.")

## 3. Data Extraction
Load data from each record set into Pandas DataFrames for analysis. Reference record set and field IDs using the `@id` values discovered above.

In [ ]:
# Extract all data from available record sets, using their @id
dataframes = {}
record_set_ids = [rs.id for rs in dataset.record_sets()]
print('Record set @id list for extraction:', record_set_ids)

for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"\nColumns in '{rs_id}':")
        print(df.columns.tolist())
        print(df.head())
    except Exception as e:
        print(f"Failed to load records for {rs_id}: {e}")

# For demonstration, select the first record set (if any) as primary for further analysis
if record_set_ids:
    main_record_set_id = record_set_ids[0]
else:
    main_record_set_id = None

## 4. Exploratory Data Analysis (EDA)
Explore, filter, and process data. 
Choose fields using their `@id` for numeric and grouping operations, and showcase normalization, filtering, and grouping.

In [ ]:
# Typical EDA: Filtering, normalization, and grouping based on @id
if main_record_set_id and main_record_set_id in dataframes:
    df = dataframes[main_record_set_id]
    print(f"\nColumns (@id) in main record set '{main_record_set_id}':")
    cols = list(df.columns)
    print(cols)

    # Attempt to pick a numeric field based on likely field names
    numeric_candidates = [c for c in cols if 'Age' in c or 'age' in c or 'Interval' in c or 'interval' in c or 'years' in c]
    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]
        print(f"\nUsing numeric field: {numeric_field_id}")

        # Set a threshold
        threshold = 10
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"\nFiltered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id}:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    else:
        print("No numeric field found for EDA.")

    # Try to group by a key attribute (e.g. Sex, MSI_Status, or another grouping field @id)
    group_candidates = [c for c in cols if 'Sex' in c or 'sex' in c or 'MSI' in c or 'subtype' in c]
    if group_candidates and numeric_candidates:
        group_field_id = group_candidates[0]
        print(f"\nGrouping by field: {group_field_id}")
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped mean {numeric_field_id} by {group_field_id}:")
        print(grouped_df)
    else:
        print("No suitable group field found.")
else:
    print("No record set DataFrame available for EDA.")

## 5. Visualization
Visualize distributions and relationships between fields by `@id`.

Below is a sample plot of the numeric field distribution and grouped means.  Requires `matplotlib`:

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and main_record_set_id in dataframes:
    df = dataframes[main_record_set_id]
    # Numeric field and group field as used above
    if 'numeric_field_id' in locals():
        plt.figure(figsize=(8,4))
        sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
        plt.title(f"Distribution of {numeric_field_id}")
        plt.xlabel(numeric_field_id)
        plt.ylabel("Count")
        plt.show()

    # Group field plot
    if 'group_field_id' in locals() and 'grouped_df' in locals():
        plt.figure(figsize=(8,4))
        sns.barplot(x=group_field_id, y=numeric_field_id, data=grouped_df)
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.show()

## 6. Conclusion
In this notebook, you explored the FAIR² dataset using `mlcroissant`, loaded all available record sets and fields referenced by their `@id`, performed basic EDA, and visualized numeric and groupwise distributions.

**Summary:**
- Data was successfully loaded and record sets and fields accessed by their `@id`
- Numeric and categorical variables were identified and processed
- Data was normalized, filtered, grouped, and visualized
- All references to fields, columns, and record sets used `@id` for reproducibility and schema alignment

Further analysis could include advanced modeling and deeper exploration of clinicopathological predictors for colorectal cancer survivors.